# Preprocessing: Annotations → Clips

This notebook handles two preprocessing steps for Phase 2A (no_hardware):

1. **Annotation conversion** — reads Label Studio JSON exports and produces a unified `annotations.csv`
2. **Clip extraction** — uses the CSV to slice individual punch clips from the full-video pose files

**Outputs:**
- `data/metadata/no_hardware/annotations.csv` — master annotation table
- `data/clips/no_hardware/{class}/{clip_id}.npy` — individual clip files for training

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("../../")  # adjust if notebook lives elsewhere

ANNOTATIONS_ROOT = PROJECT_ROOT / "data" / "annotations" / "no_hardware"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "no_hardware"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "no_hardware"
CLIPS_DIR = PROJECT_ROOT / "data" / "clips" / "no_hardware"

ANNOTATIONS_CSV = METADATA_DIR / "annotations.csv"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Annotations root: {ANNOTATIONS_ROOT}")
print(f"Processed root:   {PROCESSED_ROOT}")
print(f"Output CSV:       {ANNOTATIONS_CSV}")
print(f"Output clips:     {CLIPS_DIR}")

## Section A — Annotation Conversion

Reads Label Studio JSON exports from `data/annotations/no_hardware/{subject}/labelstudio/*.json` and builds a unified DataFrame of all clips with metadata.

In [ ]:
def parse_video_filename(filename: str) -> dict:
    """Parse 'cross_1m_left.mp4' (with optional Label Studio hash prefix) into {class, distance_m, hand}."""
    stem = Path(filename).stem
    
    # Strip Label Studio hash prefix if present (e.g. "11506747-cross_1m_left" → "cross_1m_left")
    if "-" in stem:
        prefix, _, rest = stem.partition("-")
        # Hash prefix is hex characters only
        if all(c in "0123456789abcdef" for c in prefix.lower()):
            stem = rest
    
    match = re.match(r"(jab|cross|hook|uppercut)_(\d+)m_(left|right)", stem)
    if not match:
        raise ValueError(f"Could not parse filename: {filename}")
    return {
        "class": match.group(1),
        "distance_m": int(match.group(2)),
        "hand": match.group(3),
    }

# Test with both formats
print(parse_video_filename("cross_1m_left.mp4"))
print(parse_video_filename("11506747-cross_1m_left.mp4"))

In [ ]:
def load_labelstudio_annotations(annotations_root: Path) -> list[dict]:
    """Walk through subject folders and load all Label Studio JSON exports."""
    records = []
    subject_dirs = sorted(annotations_root.glob("subject*"))
    print(f"Found {len(subject_dirs)} subject folders")
    
    for subject_dir in subject_dirs:
        subject_id = subject_dir.name
        labelstudio_dir = subject_dir / "labelstudio"
        
        if not labelstudio_dir.exists():
            print(f"  Skipping {subject_id} — no labelstudio folder")
            continue
        
        json_files = sorted(labelstudio_dir.glob("*.json"))
        print(f"  {subject_id}: {len(json_files)} JSON files")
        
        for json_path in json_files:
            with open(json_path) as f:
                data = json.load(f)
            
            tasks = data if isinstance(data, list) else [data]
            
            for task in tasks:
                video_filename = Path(task.get("data", {}).get("video", "")).name
                
                try:
                    file_info = parse_video_filename(video_filename)
                except ValueError:
                    print(f"    Skipping unparseable filename: {video_filename}")
                    continue
                
                for annotation in task.get("annotations", []):
                    for result in annotation.get("result", []):
                        if result.get("type") != "timelinelabels":
                            continue
                        
                        value = result.get("value", {})
                        labels = value.get("timelinelabels", [])
                        ranges = value.get("ranges", [])
                        
                        if not labels or not ranges:
                            continue
                        
                        for r in ranges:
                            records.append({
                                "subject_id": subject_id,
                                "video_filename": video_filename,
                                "class": labels[0],
                                "hand": file_info["hand"],
                                "distance_m": file_info["distance_m"],
                                "start_frame": int(r["start"]),
                                "end_frame": int(r["end"]),
                            })
    return records

records = load_labelstudio_annotations(ANNOTATIONS_ROOT)
print(f"\nTotal annotations loaded: {len(records)}")

In [ ]:
def build_annotations_df(records: list[dict]) -> pd.DataFrame:
    """Build the master DataFrame with unique clip_ids."""
    df = pd.DataFrame(records)
    df = df.sort_values(["subject_id", "video_filename", "start_frame"]).reset_index(drop=True)
    
    def make_clip_ids(group):
        subj_num = group["subject_id"].iloc[0].replace("subject", "")
        return [
            f"s{subj_num}_{row['class']}_{row['distance_m']}m_{row['hand']}_c{i+1:03d}"
            for i, (_, row) in enumerate(group.iterrows())
        ]
    
    df["clip_id"] = (
        df.groupby(["subject_id", "video_filename"], group_keys=False)
          .apply(lambda g: pd.Series(make_clip_ids(g), index=g.index))
    )
    
    df["frame_count"] = df["end_frame"] - df["start_frame"] + 1
    
    df = df[[
        "clip_id", "subject_id", "video_filename", "class", "hand",
        "distance_m", "start_frame", "end_frame", "frame_count"
    ]]
    return df

df = build_annotations_df(records)
df.head()

In [ ]:
df.to_csv(ANNOTATIONS_CSV, index=False)
print(f"Saved {len(df)} annotations to {ANNOTATIONS_CSV}\n")

print("=== Class distribution ===")
print(df["class"].value_counts())

print("\n=== Subject distribution ===")
print(df["subject_id"].value_counts())

print("\n=== Distance distribution ===")
print(df["distance_m"].value_counts())

print("\n=== Hand distribution ===")
print(df["hand"].value_counts())

print("\n=== Frame count statistics ===")
print(df["frame_count"].describe())

## Section B — Clip Extraction

Use the annotations CSV to slice individual punch clips from the full-video pose files.

**Reads:** `data/processed/no_hardware/{subject}/{video_stem}_pose_norm.npy`  
**Writes:** `data/clips/no_hardware/{class}/{clip_id}.npy`

In [ ]:
def strip_labelstudio_prefix(filename: str) -> str:
    """Remove Label Studio hash prefix from filename if present."""
    stem = Path(filename).stem
    suffix = Path(filename).suffix
    
    if "-" in stem:
        prefix, _, rest = stem.partition("-")
        if all(c in "0123456789abcdef" for c in prefix.lower()):
            stem = rest
    
    return stem + suffix


def extract_clips(df: pd.DataFrame, processed_root: Path, clips_dir: Path):
    """Slice individual clips from full-video pose files and save them."""
    success = 0
    failures = []
    
    for cls in df["class"].unique():
        (clips_dir / cls).mkdir(parents=True, exist_ok=True)
    
    # Iterate using iterrows() instead of itertuples() to avoid keyword collision
    for _, row in df.iterrows():
        clean_filename = strip_labelstudio_prefix(row["video_filename"])
        video_stem = Path(clean_filename).stem
        pose_path = processed_root / row["subject_id"] / f"{video_stem}_pose_norm.npy"
        
        if not pose_path.exists():
            failures.append((row["clip_id"], f"Pose file not found: {pose_path}"))
            continue
        
        try:
            pose = np.load(pose_path)
            if row["end_frame"] >= pose.shape[0]:
                failures.append((row["clip_id"],
                    f"end_frame {row['end_frame']} exceeds video length {pose.shape[0]}"))
                continue
            
            clip = pose[row["start_frame"] : row["end_frame"] + 1]
            output_path = clips_dir / row["class"] / f"{row['clip_id']}.npy"
            np.save(output_path, clip)
            success += 1
        except Exception as e:
            failures.append((row["clip_id"], str(e)))
    
    return success, failures

success, failures = extract_clips(df, PROCESSED_ROOT, CLIPS_DIR)
print(f"Successfully extracted: {success} clips")
print(f"Failures: {len(failures)}")
for clip_id, reason in failures[:10]:
    print(f"  {clip_id}: {reason}")

In [ ]:
# Cross-check clip files on disk against CSV
print("Verifying clip files on disk vs CSV:\n")
for cls in sorted(df["class"].unique()):
    cls_dir = CLIPS_DIR / cls
    on_disk = len(list(cls_dir.glob("*.npy")))
    in_csv = (df["class"] == cls).sum()
    status = "OK" if on_disk == in_csv else "MISMATCH"
    print(f"  [{status}] {cls}: {on_disk} files, {in_csv} in CSV")

# Spot check one clip
sample_row = df.iloc[0]
sample_path = CLIPS_DIR / sample_row["class"] / f"{sample_row['clip_id']}.npy"
sample = np.load(sample_path)
print(f"\nSample clip: {sample_row['clip_id']}")
print(f"  Path:  {sample_path}")
print(f"  Shape: {sample.shape}  (frames, joints, xyz)")